# Experiment 8: Multiple Linear Regression

**Objective:** Compare multiple linear regression models (2 features vs. all features) with feature scaling, multicollinearity assessment (VIF), residual diagnostics, and model serialization with `joblib`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1) Load data & build DataFrame
diabetes = load_diabetes()
X_all = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='target')

print("Dataset shape:", X_all.shape)
print("Feature names:", list(diabetes.feature_names))
X_all.head()


In [ ]:
# 2) Helper function for Model Fitting, Evaluation, and VIF
features_two = ['bmi', 'bp']         # Model A (two features)
features_all = list(X_all.columns)   # Model B (all features)

def fit_and_evaluate(feature_list, X_df, y, random_state=42):
    X = X_df[feature_list].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y.values, test_size=0.2, random_state=random_state
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)
    
    def metrics(y_true, y_pred, p):
        mae = mean_absolute_error(y_true, y_pred)
        mse = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true, y_pred)
        n = y_true.shape[0]
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if (n - p - 1) > 0 else np.nan
        return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'Adj_R2': adj_r2}
    
    metrics_train = metrics(y_train, y_pred_train, p=X_train.shape[1])
    metrics_test  = metrics(y_test, y_pred_test, p=X_test.shape[1])
    
    coef_map = dict(zip(feature_list, model.coef_))
    residuals = y_test - y_pred_test
    
    def compute_vif(X_mat, feature_names):
        vif_values = []
        X_np = np.array(X_mat)
        p = X_np.shape[1]
        for i in range(p):
            X_i = X_np[:, i]
            X_other = np.delete(X_np, i, axis=1)
            if X_other.shape[1] == 0:
                r2 = 0.0
            else:
                lr = LinearRegression().fit(X_other, X_i)
                r2 = lr.score(X_other, X_i)
            vif = 1.0 / (1.0 - r2) if (1.0 - r2) > 1e-10 else np.inf
            vif_values.append(vif)
        return pd.DataFrame({'feature': feature_names, 'VIF': np.round(vif_values, 3)})
    
    vif_df = compute_vif(X_train_scaled, feature_list)
    
    return {
        'model': model, 'scaler': scaler, 'features': feature_list,
        'coef_map': coef_map, 'intercept': model.intercept_,
        'metrics_train': metrics_train, 'metrics_test': metrics_test,
        'X_train_scaled': X_train_scaled, 'X_test_scaled': X_test_scaled,
        'y_train': y_train, 'y_test': y_test, 'y_pred_test': y_pred_test,
        'residuals': residuals, 'vif': vif_df
    }


In [ ]:
# 3) Fit Model A (two features)
print("Fitting Model A (two features: bmi, bp)...")
res_a = fit_and_evaluate(features_two, X_all, y)
print("Intercept:", round(res_a['intercept'], 4))
print("Coefficients:", res_a['coef_map'])
print("Test metrics :", res_a['metrics_test'])
print("VIF:
", res_a['vif'])


In [ ]:
# 4) Fit Model B (all features)
print("Fitting Model B (all features)...")
res_b = fit_and_evaluate(features_all, X_all, y)
print("Intercept:", round(res_b['intercept'], 4))
print("Test metrics :", res_b['metrics_test'])
print("VIF:
", res_b['vif'])


In [ ]:
# 5) Compare Model A vs Model B
print("Comparison on TEST set:")
print(f"Model A (features={res_a['features']}): R2={res_a['metrics_test']['R2']:.4f}, AdjR2={res_a['metrics_test']['Adj_R2']:.4f}")
print(f"Model B (features=all):           R2={res_b['metrics_test']['R2']:.4f}, AdjR2={res_b['metrics_test']['Adj_R2']:.4f}")


In [ ]:
# 6) Diagnostic plots for Model B
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(res_b['y_test'], res_b['y_pred_test'], alpha=0.7)
minv = min(res_b['y_test'].min(), res_b['y_pred_test'].min())
maxv = max(res_b['y_test'].max(), res_b['y_pred_test'].max())
plt.plot([minv, maxv], [minv, maxv], linestyle='--', linewidth=1, color='red', label='ideal')
plt.xlabel('Actual target (test)')
plt.ylabel('Predicted target (test)')
plt.title('Actual vs Predicted (Model B)')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(res_b['y_pred_test'], res_b['residuals'], alpha=0.7)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Predicted (test)')
plt.ylabel('Residuals (test)')
plt.title('Residuals vs Predicted (Model B)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(res_b['residuals'], bins=20, edgecolor='k', alpha=0.7)
plt.title('Residuals distribution (Model B test set)')
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 7) Save Model B & Example Prediction
joblib.dump({'model': res_b['model'], 'scaler': res_b['scaler'], 'features': res_b['features']},
            'linear_model_b_all_features.joblib')
print("Saved Model B to 'linear_model_b_all_features.joblib'")

example_raw = np.array([X_all.iloc[0].values])
scaled_example = res_b['scaler'].transform(example_raw)
pred_example = res_b['model'].predict(scaled_example)[0]
print("Example prediction (row 0) -> Actual:", float(y.iloc[0]), "| Predicted:", round(float(pred_example), 2))
